# TP3 - Premier pipeline de machine learning sur Titanic

Dans ce TP, vous allez passer de l'exploration de données à la construction d'un **premier pipeline simple de machine learning**.

Nous allons travailler uniquement sur le dataset **Titanic**, déjà étudié au TP précédent. Cette fois, l'objectif n'est plus seulement de comprendre les données, mais de les utiliser pour entraîner un modèle capable de **prédire la survie d'un passager**.

Le TP suit une progression classique en apprentissage supervisé :

- choisir les colonnes utiles ;
- préparer les données ;
- séparer les données en entraînement et test ;
- entraîner un modèle ;
- faire des prédictions ;
- mesurer la qualité des résultats ;
- comparer deux approches simples.

Le but n'est pas d'obtenir un modèle parfait, mais de **comprendre la logique générale d'un pipeline de classification**.


## Consignes de travail

- Exécutez les cellules **dans l'ordre**.
- Complétez les cellules de code sans tout réécrire inutilement.
- Lisez bien les textes d'accompagnement avant chaque partie.
- Quand une question demande une interprétation, rédigez une **réponse courte, claire et en français**.
- Regardez les sorties produites par le notebook avant de répondre.
- Retenez qu'en machine learning, un bon score ne suffit pas : il faut aussi comprendre **comment** le modèle se trompe.


## Outil principal : scikit-learn

Pour ce TP, nous allons utiliser la bibliothèque **`scikit-learn`**, souvent abrégée en **`sklearn`**.

C'est une bibliothèque Python très utilisée pour le machine learning classique. Elle fournit :

- des outils pour séparer les données en train/test ;
- des modèles déjà implémentés ;
- des fonctions pour calculer des métriques ;
- des outils de visualisation comme la matrice de confusion.

Autrement dit, `sklearn` va nous permettre de nous concentrer sur la **logique du pipeline**, sans avoir à programmer nous-mêmes les algorithmes.


## Plan du TP

1. Rechargement et préparation du dataset
2. Préparation des variables pour le modèle
3. Séparation train / test
4. Entraînement d'un premier modèle
5. Prédictions
6. Évaluation du modèle
7. Matrice de confusion et lecture des erreurs
8. Comparaison de plusieurs modèles
9. Interprétation des résultats
10. Synthèse finale


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

plt.rcParams["figure.figsize"] = (8, 4)


---
## Partie 1 - Rechargement et préparation du dataset

On repart du dataset Titanic vu au TP2, mais avec un nouvel objectif : obtenir une version du tableau qui pourra être utilisée par un modèle de machine learning.

À ce stade, il ne s'agit pas encore d'entraîner un modèle. Il faut d'abord :

- recharger proprement les données ;
- rappeler les colonnes disponibles ;
- sélectionner celles qui semblent les plus utiles ;
- écarter les colonnes trop textuelles ou trop difficiles à exploiter dans un premier essai.

C'est une étape importante, car la qualité du pipeline dépend beaucoup du choix des données en entrée.


In [ ]:
# Chargement du dataset Titanic
# Le fichier peut être lu depuis TP/TP2 ou depuis le dossier courant si nécessaire.

csv_path = Path("TP/TP2/titanic-dataset.csv")

if not csv_path.exists():
    csv_path = Path("titanic-dataset.csv")

df = pd.read_csv(csv_path)

print("Dimensions du dataset :", df.shape)


### Rappel utile

Avant de lancer un modèle, il faut toujours se poser quelques questions simples :

- quelles colonnes décrivent les passagers ?
- quelle colonne représente la sortie à prédire ?
- quelles colonnes sont numériques ?
- quelles colonnes sont catégorielles ?
- quelles colonnes sont trop complexes ou trop bruitées pour une première version ?


In [ ]:
# Exercice 1 - Premier rappel sur le dataset
# TODO 1 : affichez les 5 premières lignes du DataFrame
# TODO 2 : affichez la liste des colonnes
# TODO 3 : affichez les informations générales avec la méthode adaptée


### Choix de colonnes pour un premier modèle

Dans un premier pipeline, on ne cherche pas à utiliser toutes les colonnes possibles.

On préfère souvent commencer avec :

- quelques colonnes bien comprises ;
- peu de valeurs manquantes ;
- un sens clair par rapport au problème ;
- une structure facile à préparer.

L'idée est de construire un premier pipeline **simple, lisible et fonctionnel**.


In [ ]:
# Exercice 2 - Choix des colonnes utiles
# Nous n'allons pas garder toutes les colonnes.
# TODO 1 : créez une liste nommée colonnes_utiles contenant par exemple :
# ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
# TODO 2 : créez un nouveau DataFrame df_ml ne contenant que ces colonnes
# TODO 3 : affichez les premières lignes de df_ml


**Question 1** : Pourquoi ne garde-t-on pas toutes les colonnes du dataset pour ce premier modèle ?

_Votre réponse ici._


### Features et target

Dans un problème supervisé, on distingue toujours :

- les **features** : les informations que le modèle reçoit en entrée ;
- la **target** : la valeur que le modèle doit apprendre à prédire.

Ici, l'objectif du modèle est de prédire si un passager a survécu ou non.

Avant même d'entraîner quoi que ce soit, il est utile de regarder si la target est équilibrée ou non. Si une classe est beaucoup plus fréquente que l'autre, cela peut influencer la manière d'interpréter les résultats.


In [ ]:
# Exercice 3 - Features et target
# TODO 1 : identifiez la colonne target
# TODO 2 : affichez le nombre de passagers par valeur de la target
# TODO 3 : tracez un diagramme en barres de la target
# TODO 4 : ajoutez un titre et les labels


**Question 2** : Quelle est la target choisie et pourquoi s'agit-il d'un problème de classification ?

_Votre réponse ici._


---
## Partie 2 - Préparation des variables pour le modèle

Un modèle de machine learning ne peut pas être entraîné directement sur n'importe quel tableau.

Il faut vérifier que :

- les colonnes utiles sont bien présentes ;
- les valeurs manquantes sont traitées ;
- les variables catégorielles sont transformées en format numérique ;
- les données sont séparées proprement entre entrées (`X`) et sortie (`y`).

Autrement dit, on va transformer le dataset en une version **compréhensible par un algorithme**.


### Valeurs manquantes

La plupart des modèles de `sklearn` n'acceptent pas les `NaN` directement.

Il faut donc repérer les colonnes incomplètes et choisir une stratégie simple. Dans ce TP, on reste volontairement dans une approche accessible : remplacement par une valeur statistique simple comme la moyenne ou la médiane.


In [ ]:
# Exercice 4 - Valeurs manquantes
# TODO 1 : calculez le nombre de valeurs manquantes dans chaque colonne de df_ml
# TODO 2 : triez le résultat du plus grand au plus petit
# TODO 3 : observez quelles colonnes doivent être traitées


In [ ]:
# Exercice 5 - Traitement simple des valeurs manquantes
# TODO 1 : remplacez les NaN de Age par la médiane ou la moyenne de Age
# TODO 2 : remplacez les NaN éventuels de Fare par la médiane ou la moyenne de Fare
# TODO 3 : vérifiez qu'il ne reste plus de NaN dans ces colonnes


**Question 3** : Pourquoi est-il nécessaire de traiter les valeurs manquantes avant l'entraînement ?

_Votre réponse ici._


### Séparer `X` et `y`

Une fois le tableau nettoyé, on sépare :

- `X` : les colonnes explicatives, c'est-à-dire les informations fournies au modèle ;
- `y` : la colonne cible à prédire.

Cette séparation est centrale dans tous les pipelines `sklearn`.


In [ ]:
# Exercice 6 - Séparer features et target
# TODO 1 : créez X avec toutes les colonnes explicatives
# TODO 2 : créez y avec la target
# TODO 3 : affichez les dimensions de X et de y


### Encodage des variables catégorielles

Les colonnes comme `Sex` ou `Embarked` contiennent du texte. Or les modèles classiques de `sklearn` attendent des colonnes numériques.

Une solution simple consiste à utiliser **`pd.get_dummies(...)`**, qui transforme une variable catégorielle en plusieurs colonnes binaires.

Exemple de logique générale : une colonne contenant plusieurs catégories devient plusieurs colonnes indiquant la présence ou non de chaque catégorie.


In [ ]:
# Exercice 7 - Encodage des variables catégorielles
# Les colonnes comme Sex ou Embarked doivent être transformées en colonnes numériques.
# TODO 1 : utilisez pd.get_dummies(...) sur X
# TODO 2 : stockez le résultat dans X_encoded
# TODO 3 : affichez les premières lignes de X_encoded
# TODO 4 : affichez la liste des colonnes de X_encoded


**Question 4** : Pourquoi faut-il encoder les variables catégorielles comme `Sex` et `Embarked` ?

_Votre réponse ici._


---
## Partie 3 - Séparation des données

Avant d'entraîner un modèle, il faut séparer les données en deux groupes :

- le **jeu d'entraînement** (`train`) : il sert à apprendre ;
- le **jeu de test** (`test`) : il sert à évaluer le modèle sur des données qu'il n'a pas vues pendant l'apprentissage.

Cette séparation est indispensable si l'on veut mesurer honnêtement les performances du modèle.


### Pourquoi utiliser `train_test_split` de `sklearn` ?

`sklearn` propose une fonction très utilisée : **`train_test_split`**.

Elle permet de découper automatiquement les données en deux ensembles, avec une taille choisie à l'avance.

Dans ce TP, on prendra un découpage simple :

- `80 %` pour l'entraînement ;
- `20 %` pour le test.

On fixera aussi `random_state=42` pour que tout le monde obtienne le même découpage.


In [ ]:
# Exercice 8 - Séparation train / test
# TODO 1 : utilisez train_test_split sur X_encoded et y
# TODO 2 : utilisez test_size=0.2
# TODO 3 : fixez random_state=42 pour obtenir des résultats reproductibles
# TODO 4 : utilisez stratify=y pour conserver la proportion des classes
# TODO 5 : stockez les résultats dans X_train, X_test, y_train, y_test


In [ ]:
# Exercice 9 - Taille des sous-ensembles
# TODO 1 : affichez les dimensions de X_train et X_test
# TODO 2 : affichez les dimensions de y_train et y_test
# TODO 3 : comparez la répartition de la target dans y_train et y_test


**Question 5** : Quel est le rôle du jeu d'entraînement et du jeu de test ?

_Votre réponse ici._


**Question 6** : Pourquoi l'argument `stratify=y` est-il utile ici ?

_Votre réponse ici._


---
## Partie 4 - Entraînement d'un premier modèle

Nous allons commencer par un premier modèle très classique de `sklearn` pour la classification : **`LogisticRegression`**.

Malgré son nom, la régression logistique est bien un **modèle de classification**. Elle sert ici à estimer la probabilité qu'un passager appartienne à la classe `Survived = 1`.

L'idée n'est pas encore d'entrer dans le détail mathématique, mais de comprendre la logique suivante :

1. on crée le modèle ;
2. on l'entraîne avec `.fit(...)` ;
3. il apprend des régularités à partir des données d'entraînement.


In [ ]:
# Exercice 10 - Créer le premier modèle
# TODO 1 : créez un modèle LogisticRegression
# TODO 2 : pensez à fixer max_iter pour éviter certains avertissements
# TODO 3 : stockez ce modèle dans une variable nommée modele_lr


In [ ]:
# Exercice 11 - Entraîner le modèle
# TODO : entraînez modele_lr sur X_train et y_train avec la bonne méthode


**Question 7** : Sur quelles données le modèle apprend-il réellement ?

_Votre réponse ici._


---
## Partie 5 - Prédictions

Le modèle est maintenant entraîné. On peut donc l'utiliser pour produire des prédictions sur le jeu de test.

Cette étape est essentielle : on veut savoir comment le modèle se comporte sur des données qu'il **n'a pas utilisées pour apprendre**.

Dans `sklearn`, on utilise généralement la méthode **`.predict(...)`** pour obtenir les classes prédites.


In [ ]:
# Exercice 12 - Prédictions sur le jeu de test
# TODO 1 : faites des prédictions avec modele_lr sur X_test
# TODO 2 : stockez-les dans y_pred_lr
# TODO 3 : affichez les 10 premières prédictions


In [ ]:
# Exercice 13 - Comparer les prédictions et les vraies valeurs
# TODO 1 : créez un DataFrame comparaison avec au moins deux colonnes :
# 'vraie_valeur' et 'prediction_lr'
# TODO 2 : affichez les 15 premières lignes de ce tableau
# TODO 3 : repérez déjà quelques erreurs éventuelles


**Question 8** : Que signifie une erreur de prédiction dans ce contexte ?

_Votre réponse ici._


---
## Partie 6 - Évaluation du modèle

Une fois les prédictions calculées, il faut mesurer leur qualité.

`sklearn` fournit plusieurs métriques utiles :

- **accuracy** : proportion totale de bonnes prédictions ;
- **precision** : parmi les positifs prédits, combien sont corrects ;
- **recall** : parmi les positifs réels, combien ont été retrouvés ;
- **F1-score** : compromis entre précision et rappel.

Ces métriques ne racontent pas exactement la même chose. C'est pour cela qu'il est intéressant d'en regarder plusieurs.


In [ ]:
# Exercice 14 - Accuracy globale
# TODO 1 : calculez l'accuracy de modele_lr sur y_test et y_pred_lr
# TODO 2 : affichez ce score


In [ ]:
# Exercice 15 - Precision, recall et F1-score
# TODO 1 : calculez la précision
# TODO 2 : calculez le rappel
# TODO 3 : calculez le F1-score
# TODO 4 : affichez les trois métriques


In [ ]:
# Exercice 16 - Rapport de classification
# TODO : affichez le classification_report sur y_test et y_pred_lr


**Question 9** : Pourquoi l'accuracy seule ne suffit-elle pas toujours pour juger un modèle ?

_Votre réponse ici._


---
## Partie 7 - Matrice de confusion et lecture des erreurs

L'étape précédente donne des scores globaux. Mais pour comprendre **comment** le modèle se trompe, on utilise souvent la **matrice de confusion**.

Elle permet de comparer :

- les vraies classes ;
- les classes prédites.

On peut alors repérer les bonnes prédictions et les différents types d'erreurs.

`sklearn` propose à la fois :

- une fonction pour calculer la matrice ;
- un outil pour l'afficher sous forme de graphique.


In [ ]:
# Exercice 17 - Matrice de confusion
# TODO 1 : calculez la matrice de confusion entre y_test et y_pred_lr
# TODO 2 : affichez-la


In [ ]:
# Exercice 18 - Affichage graphique de la matrice de confusion
# TODO 1 : utilisez ConfusionMatrixDisplay pour afficher la matrice
# TODO 2 : ajoutez un titre au graphique
# TODO 3 : affichez le graphique


**Question 10** : Rappelez la signification de : vrai positif, faux positif, vrai négatif, faux négatif.

_Votre réponse ici._


**Question 11** : Quel type d'erreur vous semble le plus gênant ici ? Répondez en quelques phrases.

_Votre réponse ici._


---
## Partie 8 - Comparaison de plusieurs modèles

Un résultat n'a pas beaucoup de sens si on ne le compare à rien.

Nous allons donc tester un second modèle simple de `sklearn` : **`DecisionTreeClassifier`**.

L'idée n'est pas de faire un cours complet sur les arbres de décision, mais de montrer qu'il existe plusieurs approches possibles pour le même problème.

Ensuite, on comparera les scores obtenus par les deux modèles.


In [ ]:
# Exercice 19 - Second modèle
# TODO 1 : créez un modèle DecisionTreeClassifier
# TODO 2 : fixez random_state=42
# TODO 3 : stockez ce modèle dans une variable nommée modele_tree


In [ ]:
# Exercice 20 - Entraîner et prédire avec le second modèle
# TODO 1 : entraînez modele_tree sur X_train et y_train
# TODO 2 : faites des prédictions sur X_test
# TODO 3 : stockez-les dans y_pred_tree


In [ ]:
# Exercice 21 - Comparer les scores des deux modèles
# TODO 1 : calculez l'accuracy du modèle arbre
# TODO 2 : créez un petit DataFrame de comparaison avec les scores des deux modèles
# TODO 3 : affichez ce tableau
# TODO 4 : tracez un diagramme en barres comparant les accuracies


In [ ]:
# Exercice 22 - Comparer aussi d'autres métriques
# TODO 1 : calculez precision, recall et f1-score pour le modèle arbre
# TODO 2 : comparez-les rapidement avec ceux du modèle de régression logistique


**Question 12** : Quel modèle semble le plus performant ? Sur quels critères vous appuyez-vous ?

_Votre réponse ici._


---
## Partie 9 - Interprétation des résultats

Les scores sont utiles, mais ils ne disent pas tout.

Pour progresser en machine learning, il faut aussi apprendre à lire les erreurs du modèle :

- sur quels passagers le modèle se trompe-t-il ?
- les erreurs semblent-elles aléatoires ou répétitives ?
- certaines situations paraissent-elles plus difficiles à prédire que d'autres ?

Cette partie sert à prendre un peu de recul sur le pipeline construit.


In [ ]:
# Exercice 23 - Observer quelques erreurs du modèle
# TODO 1 : ajoutez dans le DataFrame comparaison une colonne indiquant si la prédiction LR est correcte ou non
# TODO 2 : affichez uniquement quelques lignes mal classées
# TODO 3 : observez s'il y a des profils difficiles à prédire


**Question 13** : Quelles limites voyez-vous dans ce premier pipeline ?

_Votre réponse ici._


**Question 14** : Quelles améliorations pourrait-on apporter pour essayer d'obtenir de meilleurs résultats ?

_Votre réponse ici._


---
## Partie 10 - Synthèse finale

Vous avez maintenant parcouru les grandes étapes d'un pipeline de classification avec `sklearn`.

L'objectif de cette dernière partie est de reformuler clairement ce que vous avez fait :

- quelle target vous avez choisie ;
- quelles features vous avez utilisées ;
- comment les données ont été préparées ;
- quels modèles ont été comparés ;
- et comment vous interprétez les résultats.


**1. Quelle est la target choisie ?**

_Votre réponse ici._


**2. Quelles features ont été utilisées ?**

_Votre réponse ici._


**3. Quelles étapes de préparation ont été nécessaires avant l'entraînement ?**

_Votre réponse ici._


**4. Quels modèles ont été testés ?**

_Votre réponse ici._


**5. Lequel fonctionne le mieux ?**

_Votre réponse ici._


**6. Quelles sont les principales erreurs observées ?**

_Votre réponse ici._


**7. Le modèle vous semble-t-il exploitable ? Pourquoi ?**

_Votre réponse ici._


**8. Quelles améliorations seraient possibles ?**

_Votre réponse ici._


## Fin du TP

Si vous avez terminé :

- relisez vos réponses ;
- vérifiez que toutes les cellules importantes ont bien été exécutées ;
- assurez-vous de bien comprendre le rôle de `sklearn` dans le pipeline ;
- retenez qu'un pipeline de machine learning est une suite d'étapes cohérentes, pas seulement un appel à un modèle.
